In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

#dir = "/cluster/projects/bhklab/Users/julian/PMLB"
dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)
sys.path.insert(0, dir)
from workflow.notebooks.utils.early_helpers import *

np.random.seed(101)

In [5]:
# load in files
atc = pd.read_csv("data/procdata/model_cg2/atc.csv", index_col=0)
rna = pd.read_csv("data/procdata/model_cg2/rna.csv", index_col=0)
cnv = pd.read_csv("data/procdata/model_cg2/cnv.csv", index_col=0)
mut = pd.read_csv("data/procdata/model_cg2/mut.csv", index_col=0)
pheno = pd.read_csv("data/procdata/model_cg2/meta.csv", index_col=0)

# get targest
dt = pheno["doubling_rate"]

# log normalize RNA counts
rna = np.log2(rna + 1)

# scale CNV to -1:1 (currently -2:2)
cnv = cnv/2

# binarize mutations
mut = (mut > 0).astype(int)

# Feature Selection via LASSO Selection Distribution

In [ ]:
al = run_lasso(atc, dt, "4-CommonGene2/atac_", repeats=10, features=True)
rl = run_lasso(rna, dt, "4-CommonGene2/rna_", repeats=10, features=True)
cl = run_lasso(cnv, dt, "4-CommonGene2/cnv_", repeats=10, features=True)
ml = run_lasso(mut, dt, "4-CommonGene2/mut_", repeats=10, features=True)

np.savez('data/results/data/4-CommonGene2/lasso_features.npz', a=al, r=rl, c=cl, m=ml)

### Elastic Net Models

In [ ]:
data = np.load("data/results/data/4-CommonGene2/lasso_features.npz")
al = data['a']
rl = data['r']
cl = data['c']
ml = data['m']

al.head()

In [ ]:
a_stable = al[al['frequency'] >= 0.6].index.tolist()
print(a_stable.shape[0])

r_stable = rl[rl['frequency'] >= 0.6].index.tolist()
print(r_stable.shape[0])

c_stable = cl[cl['frequency'] >= 0.6].index.tolist()
print(c_stable.shape[0])

m_stable = ml[ml['frequency'] >= 0.6].index.tolist()
print(m_stable.shape[0])

In [ ]:
# keep features that are selected by at least two modalities
dfs = [a_stable, r_stable, c_stable, m_stable]
row_counts = Counter()
for df in dfs:
    row_counts.update(df.index)
common_features = {row for row, count in row_counts.items() if count >= 2}
len(common_features)

In [ ]:
# subset omics matrices for selected features
omics = [atc, rna, cnv, mut]
selected_features = [
    df.loc[:, df.columns.intersection(common_features)]
    for df in omics
]
atc, rna, cnv, mut = selected_features

In [ ]:
print(atc.shape)
print(rna.shape)
print(cnv.shape)
print(mut.shape)

# Early Fusion without Cancer Type

In [ ]:
# scale ATAC, RNA, and CNV
continuous_df = pd.concat([atc, rna, cnv], axis=1)
scaler = StandardScaler()
continuous_scaled = pd.DataFrame(scaler.fit_transform(continuous_df),
                                 index=continuous_df.index,
                                 columns=continuous_df.columns)

# combine dataframes for early fusion
X = pd.concat([continuous_scaled, mut], axis=1)
assert (X.index == dt.index).all(), "sample mismatch"

In [ ]:
run_elastic_net(X, dt, "4-CommonGene2/ef_all")
run_random_forest(X, dt, "4-CommonGene2/ef_all")